# 06 — Sensitivity check: guideline rule with an upper age limit
The primary analyses used the rule *"refer if age ≥ 35 and BMI ≥ 25"* with **no upper age limit**. The USPSTF 2021 recommendation it is based
on covers adults **aged 35–70**, so a reviewer could fairly ask whether the comparison changes when the rule stops at 70.

This is a **sensitivity check**, added after the main results. The primary results and criteria are unchanged. The models are the saved
ones, and the same metric and bootstrap (5,000 resamples, seed 42) are used as in notebooks 03 and 05.

In [1]:
import sys, json
sys.path.append("..")

import numpy as np
import pandas as pd
import joblib

from src.data import load_clean, train_test, TARGET, PROJECT_ROOT
from src.models import GuidelineRule, sensitivity_at_rate
from src.nhanes import VALIDATION_CSV

meta = json.loads((PROJECT_ROOT / "models" / "metadata.json").read_text())
model = joblib.load(PROJECT_ROOT / meta["models"]["no_lab"]["file"])
features = meta["models"]["no_lab"]["features"]
_, pima_test = train_test(load_clean())
nhanes = pd.read_csv(VALIDATION_CSV, index_col=0)

for name, df in [("Pima test", pima_test), ("NHANES", nhanes)]:
    print(f"{name}: {len(df)} women, {(df['Age'] > 70).sum()} over 70 "
          f"({(df['Age'] > 70).mean():.1%}), of whom {int(df.loc[df['Age'] > 70, TARGET].sum())} have diabetes")

Pima test: 154 women, 0 over 70 (0.0%), of whom 0 have diabetes
NHANES: 5298 women, 706 over 70 (13.3%), of whom 147 have diabetes


In [2]:
def compare(df, rule, n_boot=5000, seed=42):
    y = df[TARGET].to_numpy()
    r = rule.predict(df).astype(float)
    s = model.predict_proba(df[features])[:, 1]
    rate = r.mean()
    point = sensitivity_at_rate(y, s, rate) - sensitivity_at_rate(y, r, rate)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        rb = r[i].mean()
        diffs.append(sensitivity_at_rate(y[i], s[i], rb) - sensitivity_at_rate(y[i], r[i], rb))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"rule refers": f"{rate:.1%}", "rule catches": f"{sensitivity_at_rate(y, r, rate):.3f}",
            "model catches": f"{sensitivity_at_rate(y, s, rate):.3f}",
            "model − rule (95% CI)": f"{point:+.3f} ({lo:+.3f} to {hi:+.3f})", "CI excludes 0": bool(lo > 0 or hi < 0)}

rows = []
for data_name, df in [("Pima test", pima_test), ("NHANES", nhanes)]:
    for rule_name, rule in [("age ≥ 35 & BMI ≥ 25 (primary)", GuidelineRule()),
                            ("age 35–70 & BMI ≥ 25 (USPSTF range)", GuidelineRule(max_age=70))]:
        rows.append({"data": data_name, "rule": rule_name, **compare(df, rule)})
pd.DataFrame(rows).set_index(["data", "rule"])

rule refers rule catches  \
data      rule                                                           
Pima test age ≥ 35 & BMI ≥ 25 (primary)             31.2%        0.519   
          age 35–70 & BMI ≥ 25 (USPSTF range)       31.2%        0.519   
NHANES    age ≥ 35 & BMI ≥ 25 (primary)             51.8%        0.778   
          age 35–70 & BMI ≥ 25 (USPSTF range)       43.1%        0.548   

                                              model catches  \
data      rule                                                
Pima test age ≥ 35 & BMI ≥ 25 (primary)               0.463   
          age 35–70 & BMI ≥ 25 (USPSTF range)         0.463   
NHANES    age ≥ 35 & BMI ≥ 25 (primary)               0.847   
          age 35–70 & BMI ≥ 25 (USPSTF range)         0.747   

                                                   model − rule (95% CI)  \
data      rule                                                             
Pima test age ≥ 35 & BMI ≥ 25 (primary)        -0.056 (-0.143 to +0.060)   
          age 35–70 & BMI ≥ 25 (USPSTF range)  -0.056 (-0.143 to +0.060)   
NHANES    age ≥ 35 & BMI ≥ 25 (primary)        +0.069 (+0.032 to +0.109)   
          age 35–70 & BMI ≥ 25 (USPSTF range)  +0.200 (+0.143 to +0.255)   

                                               CI excludes 0  
data      rule                                                
Pima test age ≥ 35 & BMI ≥ 25 (primary)                False  
          age 35–70 & BMI ≥ 25 (USPSTF range)          False  
NHANES    age ≥ 35 & BMI ≥ 25 (primary)                 True  
          age 35–70 & BMI ≥ 25 (USPSTF range)           True

**What this shows**
- **Pima test:** no woman is over 70, so the age limit changes nothing. The Pima results stand exactly as reported.
- **NHANES:** 706 women (13 %) are over 70, and they include **147 of the 451 cases (33 %)**. The 35–70 rule can't refer any of them, so it
  catches only **55 %** of cases (78 % without the limit). At the 35–70 rule's referral rate (43 %), the no-lab model catches 75 %, an
  advantage of **+20 points (95 % CI +14 to +26)**, against +6.9 with the primary rule.
- **Interpretation:** the primary analysis compared the model with the **more generous** version of the rule, the one that catches more
  cases. So the main comparison was **conservative**. Against the guideline's actual age range, the model's advantage in NHANES is much larger.
- *Caveat:* this check was added after the main results, so it doesn't replace the pre-set comparison. It shows that the primary choice of
  baseline didn't flatter the model.